# Popularity based модель

Мы будем работать с набором данных Movie Lens. Он содержит идентификаторы для каждого фильма и пользователя, который его смотрел, а также оценку, которую пользователь поставил фильму. В датасете представлено 25 000 095 оценок фильмов от 162 541 пользователя со шкалой оценок от 0.5 до 5.0.

In [21]:
import pandas as pd
import numpy as np
import re

Из этого набора нам понадобится два файла:

* данные о фильмах (movies);
* данные о выставленных оценках (ratings).

Объединим их:

In [22]:
movie = pd.read_csv('Data/movie.csv')
ratings = pd.read_csv('Data/ratings.csv')

Дата сет:

* userId — id пользователя;
* movieId — id фильма;
* rating — выставленный пользователем рейтинг для фильма;
* timestamp — время выставления рейтинга;
* title — название фильма;
* genres — жанры, к которым относится фильм.

In [23]:
movie_df = pd.merge(ratings, movie, how='left', on='movieId')
display(movie_df.head())

,userId,movieId,rating,timestamp,title,genres
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),Adventure|Children|Fantasy
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


Подсчитаем, сколько раз каждый фильм встречается в наборе данных. Отметим среди перечисленных ниже фильмов те, что встречаются в топ-5 по популярности.

In [24]:
film_counts = movie_df['title'].value_counts()
top_5 = film_counts.head(5)
display(top_5)

title
Pulp Fiction (1994)                 67310
Forrest Gump (1994)                 66172
Shawshank Redemption, The (1994)    63366
Silence of the Lambs, The (1991)    63299
Jurassic Park (1993)                59715
Name: count, dtype: int64

Отлично, мы нашли самые востребованные фильмы. Однако если фильм посмотрело много людей, это ещё не значит, что он им понравился. Чтобы понять, как зритель на самом деле относится к фильму, нужны более чёткие данные. К счастью, в наборе данных Movie Lens есть оценки каждого из зрителей.

* Найдем средний рейтинг для каждого из фильмов.
* Найдем фильмы с наивысшим средним рейтингом.
* Введем в качестве результата фильм, занимающий последнее место среди фильмов с наивысшим рейтингом, если предварительно отсортировать их по алфавитному порядку.

In [25]:
mean_rating = movie_df.groupby('title')['rating'].mean().reset_index()
mean_rating

,title,rating
0,#chicagoGirl: The Social Network Takes on a Di...,3.666667
1,$ (Dollars) (1971),2.833333
2,$5 a Day (2008),2.871795
3,$9.99 (2008),3.009091
4,$ellebrity (Sellebrity) (2012),2.000000
...,...,...
26724,À propos de Nice (1930),3.125000
26725,Árido Movie (2005),2.000000
26726,Åsa-Nisse - Wälkom to Knohult (2011),1.500000
26727,Üvegtigris (2001),3.000000


In [26]:
# Сортируем по рейтингу
top_mean_rating = mean_rating.sort_values(by='rating', ascending=False)
# Определяем максимальный рейтинг среди среднего значения рейтинга
max_rating = top_mean_rating['rating'].iloc[0]
# Сортируем фильмы по максимальному рейтингу
top_rated_movies = top_mean_rating[top_mean_rating['rating'] == max_rating]
# Сортируем фильмы по названию
top_rated_movies_sorted = top_rated_movies.sort_values(by='title')
# Выбираем последний фильм в этом списке
last_top_movie = top_rated_movies_sorted.tail(1)
# Вывод
display(last_top_movie)

,title,rating
26482,Yonkers Joe (2008),5.0


Выше мы использовали два самых простых метода для создания неперсонализированных рекомендаций. Однако у них обоих есть свои недостатки: 

- поиск наиболее часто просматриваемых фильмов не учитывает того, насколько фильм нравится аудитории, 
- а поиск среднего рейтинга может вывести в рекомендуемые фильмы малоизвестные специфические картины с одной-двумя оценками.

Чтобы решить эти проблемы, объединим два подхода и будем искать средний рейтинг только для фильмов, которые были оценены более 50 раз.

In [27]:
rating_counts = movie_df.groupby('title')['rating'].count().reset_index(name='count')

merged = mean_rating.merge(rating_counts, on='title')
display(merged)

rates_fifty_times = merged[merged['count'] > 50]

display(rates_fifty_times.shape[0])

,title,rating,count
0,#chicagoGirl: The Social Network Takes on a Di...,3.666667,3
1,$ (Dollars) (1971),2.833333,24
2,$5 a Day (2008),2.871795,39
3,$9.99 (2008),3.009091,55
4,$ellebrity (Sellebrity) (2012),2.000000,2
...,...,...,...
26724,À propos de Nice (1930),3.125000,4
26725,Árido Movie (2005),2.000000,1
26726,Åsa-Nisse - Wälkom to Knohult (2011),1.500000,2
26727,Üvegtigris (2001),3.000000,1


10472

Построем простейшую рекомендацию: возьмем фильмы, которые смотрели более 50 раз, и найдим среди них фильм с наивысшей средней оценкой. В качестве ответа запишем название этого фильма без артикля и года выхода на экран.

In [28]:
most_popular = rates_fifty_times.sort_values(by='rating', ascending=False).iloc[0]['title']
# Убираем год
recommended = re.sub(r'\s+\(\d{4}\)$', '', most_popular)
display(recommended)

'Shawshank Redemption, The'

# Articles sharing and reading from CI&T DeskDrop

Датасет включает в себя собранные за один год логи DeskDrop — платформы для внутренних коммуникаций, разработанной CI&T и ориентированной на компании, использующие Google Workspace (Google G Suite).

В датасете содержится около 73 тысяч записей о взаимодействии пользователей с более чем тремя тысячами публичных статей, размещённых на платформе.

Данные включают в себя два файла:

* shared_articles.csv;
* users_interactions.csv.

In [227]:
import pandas as pd
import numpy as np
import math
from scipy.linalg import svd

from lightfm import LightFM
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k
from scipy.sparse import csr_matrix

Начнём работать с файлом __shared_articles.csv__. Он содержит информацию о статьях, опубликованных на платформе DeskDrop.

Каждая статья содержит:

* дата публикации (временная метка),
* исходный URL-адрес,
* заголовок,
* содержание в виде обычного текста,
* язык статьи (португальский — pt или английский — en),
* информация о пользователе, который поделился статьёй (автор).

In [91]:
art_df = pd.read_csv('Data/shared_articles.csv')
display(art_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3122 entries, 0 to 3121
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   timestamp        3122 non-null   int64 
 1   eventType        3122 non-null   object
 2   contentId        3122 non-null   int64 
 3   authorPersonId   3122 non-null   int64 
 4   authorSessionId  3122 non-null   int64 
 5   authorUserAgent  680 non-null    object
 6   authorRegion     680 non-null    object
 7   authorCountry    680 non-null    object
 8   contentType      3122 non-null   object
 9   url              3122 non-null   object
 10  title            3122 non-null   object
 11  text             3122 non-null   object
 12  lang             3122 non-null   object
dtypes: int64(4), object(9)
memory usage: 317.2+ KB


None

Для временной метки существует два возможных типа событий:

* CONTENT SHARED — статья была опубликована на платформе и доступна для пользователей;
* CONTENT REMOVED — статья была удалена с платформы и недоступна для дальнейших рекомендаций.

Примечание: Для простоты будем рассматривать только тип события - CONTENT SHARED.

Отфильтруем данные так, чтобы остались только объекты с типом события CONTENT SHARED. Определим сколько таких объектов получилось в таблице.

In [92]:
art_df.contentId = art_df.contentId.astype(str)
content_mask = art_df['eventType'] == 'CONTENT SHARED'

articles_df = art_df[content_mask]
display(articles_df.shape[0])

3047

Откроем второй файл — users_interactions.csv.

Предварительно преобразуем столбцы personId, contentId в таблицах к строкам. Это преобразование пригодится в дальнейшем:

In [93]:
interactions_df = pd.read_csv('Data/users_interactions.csv')
display(interactions_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72312 entries, 0 to 72311
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   timestamp    72312 non-null  int64 
 1   eventType    72312 non-null  object
 2   contentId    72312 non-null  int64 
 3   personId     72312 non-null  int64 
 4   sessionId    72312 non-null  int64 
 5   userAgent    56918 non-null  object
 6   userRegion   56907 non-null  object
 7   userCountry  56918 non-null  object
dtypes: int64(4), object(4)
memory usage: 4.4+ MB


None

In [94]:
interactions_df.personId = interactions_df.personId.astype(str)
interactions_df.contentId = interactions_df.contentId.astype(str)

В колонке eventType описаны действия, которые могли совершать пользователи при взаимодействии со статьёй:

* VIEW — просмотр,
* LIKE — лайк,
* COMMENT CREATED — комментарий,
* FOLLOW — подписка,
* BOOKMARK — добавление в закладки.

In [95]:
display(interactions_df['eventType'].head(5))

0      VIEW
1      VIEW
2      VIEW
3    FOLLOW
4      VIEW
Name: eventType, dtype: object

В первую очередь необходимо понять, как определить, что какая-то статья популярнее других. Если бы из возможных реакций у нас были только лайки или только просмотры, то статьи было бы легко ранжировать в соответствии с этими значениями. Однако у нас есть информация о различных действиях пользователя, и на её основе мы должны создать некий универсальный индекс популярности. Составим его из реакций пользователей, придав им разные веса:

In [96]:
event_type = {
    'VIEW': 1.0,
    'LIKE': 2.0,
    'BOOKMARK': 2.5,
    'FOLLOW': 3.0,
    'COMMENT CREATED': 4.0
}

Примечание: Веса здесь подобраны исходя из важности каждого действия: оставить комментарий — значит, показать наибольшую вовлечённость, а обычный просмотр, напротив, демонстрирует наименьшую вовлечённость.

Создадим признак, который будет отражать числовой вес для взаимодействия со статьёй (в соответствии с приведёнными выше весами). Вычислим среднее значение для полученного признака и округлим его до двух знаков после точки-разделителя.

In [97]:
interactions_df['article_score'] = interactions_df['eventType'].map(event_type)
mean_score = round(interactions_df['article_score'].mean(), 2)
print(mean_score)

1.24


Ранее обсуждалось, что рекомендательные системы подвержены проблеме __холодного старта__ — в таких случаях создавать рекомендации намного сложнее.

Чтобы получить хоть какую-то информацию, на которую можно будет опираться, оставим только тех пользователей, которые взаимодействовали хотя бы с пятью статьями. Определим сколько всего таких статей.

In [98]:
# Считаем количество уникальных contentId для каждого personId
user_article_counts = interactions_df.groupby('personId')['contentId'].nunique().reset_index(name='article_count')
# Оставляем пользователей с >= 5 уникальными статьями
active_users = user_article_counts[user_article_counts['article_count'] >= 5]

# Количество таких пользователей
num_active_users = active_users.shape[0]
display(num_active_users)

1140

Теперь стоит оставить только те взаимодействия, которые касаются только отфильтрованных пользователей (то есть тех, которые взаимодействовали как минимум с пятью статьями). Сколько всего таких взаимодействий?

In [99]:
# Фильтруем датафрейм
interactions_df_active_users = interactions_df[interactions_df['personId'].isin(active_users['personId'])]
num_interactions = interactions_df_active_users.shape[0]
display(num_interactions)

69868

Сейчас каждое отдельное взаимодействие пользователя со статьёй выделено в отдельную запись, то есть пользователь мог просмотреть статью, лайкнуть и прокомментировать её, и всё это отразилось в трёх действиях. Для удобства соединим все эти действия в некоторый коэффициент, который будет отражать интерес пользователя к статье. Так как каждому возможному действию мы ранее уже присвоили вес, то, по сути, нам нужно просто сложить все действия. Однако полученное число будет увеличиваться с количеством действий, и будет очень большой разброс возможных значений. В таких случаях обычно логарифмируют полученный результат с помощью функции:

In [100]:
def smooth_user_preference(x):
    return math.log(1+x, 2)

Применим упомянутое выше преобразование для логарифмирования к сумме весов для взаимодействия пользователя с каждой конкретной статьёй. Также сохраним для каждой пары «пользователь — статья» значение времени последнего взаимодействия.

Найдем среднее по признаку с получившимися временными отсечками. Округлим результат до двух знаков после точки-разделителя.

In [101]:
# Группируем по паре (personId, contentId)
agg_df = interactions_df_active_users.groupby(['personId', 'contentId']).agg({
    'article_score': 'sum',
    'timestamp': 'max'
}).reset_index()

# Применяем логарифмирование к сумме весов
agg_df['preference'] = agg_df['article_score'].apply(smooth_user_preference)

# Вычисляем среднее значение timestamp
mean_timestamp = round(agg_df['timestamp'].mean(), 2)
print(mean_timestamp)

1470605340.04


Разумеется, для того чтобы впоследствии оценить __качество построенной рекомендательной системы__, нам нужно разделить выборку на обучающую и тестовую. Так как в реальности рекомендации строятся на основе исторических данных о пользователе и контенте, сделаем разбиение на обучающую и тестовую выборки по временной отсечке.

* все записи с timestamp <= отсечка -> train
* все записи с timestamp > отсечка -> test

In [102]:
# Определим временную отсечку
cutoff_timestamp = 1475519545

# Делим данные
train_df = agg_df[agg_df['timestamp'] <= cutoff_timestamp]
test_df = agg_df[agg_df['timestamp'] > cutoff_timestamp]

# Размеры выборок
print(f"Размер train: {train_df.shape[0]} взаимодействий")
print(f"Размер test: {test_df.shape[0]} взаимодействий")

Размер train: 29325 взаимодействий
Размер test: 9781 взаимодействий


Для удобства дальнейшего измерения качества рекомендаций преобразуем данные так, чтобы получить таблицу в формате, где строка соответствует пользователю, а столбцы будут истинными предпочтениями и рекомендациями в формате списков. На место пустых ячеек поместим пустые списки.

In [103]:
final_df = (
    train_df.reset_index()
    .groupby('personId')['contentId'].agg(lambda x: list(x))
    .reset_index()
    .rename(columns={'contentId': 'true_train'})
    .set_index('personId')
)

final_df['true_test'] = (
    test_df.reset_index()
    .groupby('personId')['contentId'].agg(lambda x: list(x))
)

final_df['true_test'] = [ [] if x is np.nan else x for x in final_df['true_test'] ]
final_df.head()

,true_train,true_test
personId,,
-1007001694607905623,"[-5065077552540450930, -793729620925729327]","[-6623581327558800021, 1469580151036142903, 72..."
-1032019229384696495,"[-1006791494035379303, -1039912738963181810, -...","[-1415040208471067980, -2555801390963402198, -..."
-108842214936804958,"[-1196068832249300490, -133139342397538859, -1...","[-2780168264183400543, -3060116862184714437, -..."
-1130272294246983140,"[-1150591229250318592, -1196068832249300490, -...","[-1606980109000976010, -1663441888197894674, -..."
-1160159014793528221,"[-133139342397538859, -387651900461462767, 377...",[-3462051751080362224]


Теперь мы будем строить __popular-based-модель__, а значит, необходимо найти самые популярные статьи.

Посчитаем популярность каждой статьи как сумму всех логарифмических «оценок» взаимодействий с ней (используя только обучающую выборку). Выберим ID самой популярной статьи:

In [105]:
# Группировка по contentId
articles_popularity = (
    train_df.groupby('contentId')['preference']
    .sum()
    .reset_index()
)

# Сортировка по популярности
popularuty_sort = articles_popularity.sort_values(by='preference', ascending=False)

# ID самой популярной статьи
most_popular_article_id = popularuty_sort.iloc[0]['contentId']
print(f"ID самой популярной статьи: {most_popular_article_id}")

# Преобразуем в массив contentId, отсортированных по популярности
popular = popularuty_sort['contentId'].values

ID самой популярной статьи: -6783772548752091658


Построим систему рекомендаций. Оцении качество с помощью precision@10 для каждого пользователя (доля угаданных рекомендаций). После этого усредним результат по всем пользователям.

Для вычисления precision@10 воспользуемся следующей функцией:

In [169]:
def calc_precision(column, df):
    return (df.apply(lambda row:
            len(set(row['true_test']).intersection(set(row[column]))) /
            min(len(row['true_test']) + 0.001, 10.0),axis=1)).mean()

In [170]:
top_k = 10

final_df['popular'] = (
    final_df.true_train
    .apply(lambda x: popular[~np.isin(popular, x)][:top_k])
)

precision_metric = round(calc_precision('popular', final_df), 3)
display(f'precision metric: {precision_metric}')

'precision metric: 0.006'

# Вывод:

Качество получилось не очень высоким, но ведь и рекомендации у нас были неперсонализированными.

Примечание. Стоит отметить, что качество РС оценивается не так, как в задачах классификации: показателей выше 0.5 добиться практически невозможно, и даже результат 0.1–0.2 — индикатор высокого качества.

Далее построим матрицу, в которой по столбцам будут находиться id статей, по строкам — id пользователей, а на пересечениях строк и столбцов — оценка взаимодействия пользователя со статьёй. Если взаимодействия не было, в соответствующей ячейке должен стоять ноль. Будем работать с данными train_df.

In [128]:
train_df

,personId,contentId,article_score,timestamp,preference
0,-1007001694607905623,-5065077552540450930,1.0,1470395911,1.0
2,-1007001694607905623,-793729620925729327,1.0,1472834892,1.0
6,-1032019229384696495,-1006791494035379303,1.0,1469129122,1.0
7,-1032019229384696495,-1039912738963181810,1.0,1459376415,1.0
8,-1032019229384696495,-1081723567492738167,3.0,1464054096,2.0
...,...,...,...,...,...
39099,997469202936578234,9112765177685685246,3.0,1472479493,2.0
39100,998688566268269815,-1255189867397298842,1.0,1474567164,1.0
39101,998688566268269815,-401664538366009049,1.0,1474567449,1.0
39103,998688566268269815,6881796783400625893,1.0,1474567675,1.0


In [204]:
ratings = pd.pivot_table(
    train_df, 
    index='personId', # По строкам пользователи
    columns='contentId', # По столбцам статьи
    values='preference', # Пересечение взаимодействия со статьей ('сила взаимодействия')
).fillna(0) # Если взаимодействия не было 0

In [205]:
# Найдем оценку взаимодействия для пользователя с ID -1032019229384696495 со статьёй с ID 943818026930898372. 
# Результат округлим до двух знаков после точки-разделителя.
id_user = '-1032019229384696495'
id_article = '943818026930898372'

result = ratings.at[id_user, id_article]
display(round(result, 2))

2.32

Далее попробуем использовать memory-based подход коллаборативной фильтрации.

In [206]:
# Для ускорения преобразуем таблицу в numpy массив
ratings_array = np.asarray(ratings)
result = round(ratings_array.mean(), 3)
display(result)

0.017

In [207]:
# 1112 пользователей; 2366 статей
ratings_array.shape

(1112, 2366)

Попробуем реализовать алгоритмы коллаборативной фильтрации «с нуля». Такая практика позволит выстроить более сложную систему, чем могут предложить готовые модули (surprise). Кроме того, «ручная» реализация алгоритмов позволит лучше понять принцип их работы.

Построим матрицу схожести. Для этого вычислим все попарные коэффициенты корреляции для матрицы, полученной на предыдущем этапе. Для каждой пары учитывам только ненулевые значения (так как нулевые обозначают отсутствие взаимодействия и не интересуют нас). Выведим результат, полученный в ячейке с третьим индексом по строкам и сороковым — по столбцам. Ответ округлим до двух знаков после точки-разделителя.

In [208]:
# Присваиваем переменной ratings_m массив взаимодействий (матрицу взаимодействий в виде numpy array)
ratings_m = ratings_array

# Создаём пустую матрицу для хранения коэффициентов схожести пользователей (симметричная матрица)
# Размер матрицы — число пользователей x число пользователей
similarity_users = np.zeros((len(ratings_m), len(ratings_m)))

# Двойной цикл по парам пользователей (только по уникальным парам, чтобы не считать дважды)
for i in range(len(ratings_m) - 1):
    for j in range(i + 1, len(ratings_m)):
        
        # Создаём маску для элементов, где у обоих пользователей есть ненулевые взаимодействия (т.е. общие статьи)
        mask_uv = (ratings_m[i] != 0) & (ratings_m[j] != 0)
        
        # Извлекаем оценки по этим общим статьям для обоих пользователей
        ratings_v = ratings_m[i, mask_uv]
        ratings_u = ratings_m[j, mask_uv]
        
        # Вычисляем коэффициент корреляции Пирсона между этими оценками
        # np.corrcoef возвращает матрицу 2x2, берем элемент [0, 1] — сам коэффициент
        similarity_users[i, j] = np.corrcoef(ratings_v, ratings_u)[0, 1]
        
        # Так как матрица симметричная, значение [j, i] такое же, как [i, j]
        similarity_users[j, i] = similarity_users[i, j]

# Выводим коэффициент корреляции между пользователем с индексом 3 и пользователем с индексом 40, округленный до двух знаков
print(f'Коэфф корреляции: {round(similarity_users[3, 40], 2)}')

/Users/alexander/.pyenv/versions/3.10.14/lib/python3.10/site-packages/numpy/lib/function_base.py:520: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/Users/alexander/.pyenv/versions/3.10.14/lib/python3.10/site-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/alexander/.pyenv/versions/3.10.14/lib/python3.10/site-packages/numpy/lib/function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/alexander/.pyenv/versions/3.10.14/lib/python3.10/site-packages/numpy/lib/function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/alexander/.pyenv/versions/3.10.14/lib/python3.10/site-packages/numpy/lib/function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/Users/alexander/.pyenv/versions/3.10.14/lib/python3.10/site-packages/numpy/lib/functi

Коэфф корреляции: -0.33


Теперь мы имеем матрицы схожести пользователей. Их можно использовать для построения рекомендаций. Чтобы это сделать, надо реализовать алгоритм:

Для каждого пользователя:

1. Найти пользователей с похожестью больше 0.
2. Для каждой статьи вычислить долю пользователей (среди выделенных на первом шаге), которые взаимодействовали со статьёй.
3. Порекомендовать статьи (не более 10) с наибольшими долями со второго шага (среди тех, которые пользователь ещё не видел).

In [209]:
# Формируем таблицу взаимодействий для train — группируем по personId и собираем список контента (статей), с которым взаимодействовал пользователь
interactions = (
    train_df
    .groupby('personId')['contentId'].agg(lambda x: list(x))  # для каждого пользователя собираем список статей
    .reset_index()  # возвращаем personId в обычный столбец
    .rename(columns={'contentId': 'true_train'})  # переименовываем колонку contentId в true_train
    .set_index('personId')  # делаем personId индексом
)

# Добавляем в таблицу interactions аналогичные списки для test — статьи из тестовой выборки
interactions['true_test'] = (
    test_df
    .groupby('personId')['contentId'].agg(lambda x: list(x))  # собираем список статей из теста
)

# Обрабатываем пропуски: если для пользователя нет тестовых статей, ставим пустой список
interactions['true_test'] = [ [] if x is np.NaN else x for x in interactions['true_test'] ]

In [211]:
# Список для хранения рекомендаций по user-based методу
prediction_user_based = []

# 1 этап:
# Для каждого пользователя строим рекомендации
for i in range(len(similarity_users)):
    # Получаем булев вектор: какие пользователи схожи с этим пользователем (схожесть > 0)
    users_sim = similarity_users[i] > 0
    
    # 2 этап:
    # Если нет похожих пользователей, рекомендаций нет — добавляем пустой список
    if not any(users_sim):
        prediction_user_based.append([])
    else:
        
        # 3 этап:
        # Суммируем взаимодействия всех похожих пользователей по статьям
        summed_interactions = ratings_m[users_sim].sum(axis=0)
        # Сортируем статьи по убыванию суммы взаимодействий (популярность среди похожих пользователей)
        tmp_recommend = np.argsort(summed_interactions)[::-1]
        # Преобразуем индексы в реальные id статей с помощью interaction_matrix.columns
        tmp_recommend = ratings.columns[tmp_recommend]
        # Исключаем статьи, которые пользователь уже видел в обучении
        recommend = np.array(tmp_recommend)[~np.in1d(tmp_recommend, interactions.iloc[i]["true_train"])][:10]
        # Добавляем топ-10 рекомендаций
        prediction_user_based.append(list(recommend))

# Добавляем список рекомендаций в DataFrame с взаимодействиями
interactions['prediction_user_based'] = prediction_user_based

In [212]:
# Выводим первую рекомендацию для пользователя с индексом 35
print(f"Первая рекомендация для пользователя 35: {prediction_user_based[35][0]}")

Первая рекомендация для пользователя 35: -5148591903395022444


После того как сделаны предсказания, можно вычислить качество по метрике, которую мы определили выше:

In [213]:
precision_metric = round(calc_precision('prediction_user_based', interactions), 3)
display(f'precision metric: {precision_metric}')

'precision metric: 0.005'

Теперь реализуем рекомендательную систему с использованием SVD.

Разложим матрицу взаимодействий пользователей со статьями с помощью функции svd из модуля scipy. Найдем максимальное значение в получившейся матрице U. Результат округлим до двух знаков после точки-разделителя.

In [214]:
U, s, Vh = svd(ratings_array)
display(round(U.max(), 2))

0.71

Примечание: Значения матрицы с сингулярными числами отсортированы по убыванию. Допустим, мы хотим оставить только первые 100 компонент и получить скрытые представления размерности 100. Для этого необходимо оставить 100 столбцов в матрице U, только первые 100 значений из sigma (и сделать из них диагональную матрицу) и 100 строк в матрице Vh. Затем необходимо перемножить преобразованные матрицы.

In [215]:
# Оставляем первые 100 компонент
U_100 = U[:, :100]             # Первые 100 столбцов U
S_100 = np.diag(s[:100])       # Диагональная матрица из первых 100 сингулярных чисел (сингулярная матрица)
V_100 = Vh[:100, :]            # Первые 100 строк Vh

# Результат исходной матрицы взаимодействий, но с уменьшенным размером скрытых факторов
new_ratings = pd.DataFrame(
    U_100.dot(S_100).dot(V_100), index=ratings.index,
    columns=ratings.columns
)

In [216]:
# Найдем сумму всех элементов сингулярной матрицы
display(round(S_100.sum(), 2))

2096.43

Теперь можно сделать предсказание по полученной матрице.

Примечание: Помним, что не нужно учитывать статьи, которые уже были просмотрены пользователем.

Найдем для каждого пользователя статьи с наибольшими оценками в восстановленной матрице.

In [220]:
top_k = 10  # Количество топ-рекомендаций, которые мы хотим отобрать для каждого пользователя
predictions = []  # Список для хранения предсказанных рекомендаций для каждого пользователя

# Проходим по всем пользователям в таблице interactions (индекс — это personId)
for personId in interactions.index:
    # Берём строку из матрицы предсказанных рейтингов (new_ratings) для данного пользователя
    # Сортируем статьи по убыванию предсказанных оценок
    prediction = (new_ratings.loc[personId].sort_values(ascending=False).index.values)

    # Оставляем только те статьи, которые пользователь ещё не видел (отсекаем те, что были в true_train)
    filtered_prediction = prediction[~np.in1d(prediction, interactions.loc[personId, "true_train"])]

    # Добавляем в список предсказания топ-K статей
    predictions.append(list(filtered_prediction[:top_k]))

# Добавляем в DataFrame колонку с предсказаниями по SVD
interactions["prediction_svd"] = predictions


In [221]:
precision_metric = round(calc_precision("prediction_svd", interactions), 3)
print(precision_metric)

0.012


В итоге, были реализованы два алгоритма коллаборативной фильтрации буквально с нуля! Теперь для полноты картины реализуем на этих данных гибридную модель и посмотрим, какое качество получится. Для этого используем библиотеку LightFM.

Возьмем матрицу, подготовленную выше и преобразуем её в разреженную матрицу:

In [236]:
# Преобразуем в разреженную матрицу
ratings_matrix = csr_matrix(ratings)

# Разделим выборку на 70% train и 30% test
train, test = random_train_test_split(
    ratings_matrix, # Общая выборка
    test_percentage=0.3, # размер тестовой выборки 30%
    random_state=13
)

# Создаем модель LightFM
model = LightFM(
    loss='warp',
    random_state=13, # фиксируем значения
    learning_rate=0.05, # темп обучения
    no_components=100 # рамерность вектора для представления данных в модели
)

# Обучаем модель
model = model.fit(
    train, # обучающая выборка
    verbose=True # отображение обучения
)

# Считаем precision@10
model_metrics = precision_at_k(
    model=model,
    test_interactions=test,
    k=10
)

# Считаем среднее и округляем
mean_precision = np.mean(model_metrics)
print(f"Precision@10: {mean_precision:.2f}")

Epoch 0
Precision@10: 0.04


# Вывод: 

В данном случае модель «из коробки» показала наилучший результат, однако это совсем не показатель того, что стоит пользоваться исключительно готовыми функциями. Зная тонкости работы алгоритмов, мы можем создавать собственные гибридные системы, настраивать отдельные алгоритмы и добиваться ещё лучших результатов.

# Content based модель

Будем работать с датасетом, содержащим информацию об оценивании фильмов на платформе Netflix.

* show_id — id фильма,
* type — его тип (фильм или сериал),
* title — название,
* director — режиссер,
* cast — актерский состав,
* country — страна,
* date_added — дата добавления,
* release_year — год выхода на экраны,
* rating — рейтинг,
* duration — продолжительность,
* listened_in — жанр(-ы),
* description — описание.

In [46]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer # Векторизация текста
from sklearn.metrics.pairwise import linear_kernel # Косинусная близость

In [47]:
netflix_df = pd.read_csv('Data/netflix_titles.csv')
display(netflix_df.head(5))

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...


В первую очередь нам необходимо определить, на основании чего мы будем рассматривать близость фильмов. Выберем для этой задачи описание фильма, ведь в нём, скорее всего, содержится много информации. Однако описание — это текст. Есть много подходов к преобразованию текста в вектор, и мы будем использовать подход TF-IDF (Term Frequency-Inverse Document Frequency).

Таким образом:

* Коэффициент будет выше, если слово характерно именно для этого текста, то есть встречается в данном тексте часто, но не встречается в других текстах.
* Коэффициент будет ниже, если слово не встречается почти нигде или встречается одинаковое количество раз во всех текстах, то есть не характеризует никакой текст в отдельности.

Далее учтём стоп-слова, т.е. предлоги и другие служебные части речи, которые не несут содержательной информации, и с учётом этого определим нашу модель:

In [48]:
tf_model = TfidfVectorizer(stop_words='english')

# Заполним пропуски пустыми строками:
netflix_df['description'] = netflix_df['description'].fillna('')

# Трансформируем наши описания в матрицу:
feature_matrix = tf_model.fit_transform(netflix_df['description'])

In [49]:
# Определим колиство столбцов в матрице
display(feature_matrix)

<7787x17905 sparse matrix of type '<class 'numpy.float64'>'
	with 107187 stored elements in Compressed Sparse Row format>

Теперь необходимо вычислить косинусную близость. Можно сделать это так:

In [50]:
cosine_sim = linear_kernel(feature_matrix, feature_matrix)

Примечание: Мы используем здесь linear_kernel(), а не cosine_similarity(), так как в косинусном расстоянии в знаменателе реализуется нормировка векторов, а TF-IDF создаёт уже нормализованные векторы.

In [51]:
# Вернем индексацию и уберем дубликаты из данных
indices = pd.Series(netflix_df.index, index=netflix_df['title']).drop_duplicates()

Теперь пропишем функцию для создания рекомендаций:

In [52]:
def get_recommendations(title):
    idx = indices[title]
    # вычисляем попарные коэффициенты косинусной близости
    scores = list(enumerate(cosine_sim[idx]))
    # сортируем фильмы на основании коэффициентов косинусной близости по убыванию
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    # выбираем десять наибольших значений косинусной близости; нулевую не берём, т.к. это тот же фильм
    scores = scores[1:11]
    # забираем индексы
    ind_movie = [i[0] for i in scores]
    # возвращаем названия по индексам
    return netflix_df['title'].iloc[ind_movie]

In [53]:
netflix_df['title'].unique()

array(['3%', '7:19', '23:59', ..., 'Zulu Man in Japan',
       "Zumbo's Just Desserts", "ZZ TOP: THAT LITTLE OL' BAND FROM TEXAS"],
      dtype=object)

In [54]:
get_recommendations('Balto')

709                Balto 2: Wolf Quest
7446                           Vroomiz
1338    Chilling Adventures of Sabrina
7388                          Vampires
1770                          Dinotrux
2767                     Hold the Dark
5540                 Shanghai Fortress
4041                             Mercy
2582                       Half & Half
1365        Christmas in the Heartland
Name: title, dtype: object

# Memory based model

Для создания алгоритмов рекомендательной системы будем использовать библиотеку surprise.

```python
    pip install scikit-surprise
```

In [55]:
import pandas as pd
import numpy as np

import surprise
from surprise import Dataset
from surprise import Reader
# с помощью данного объекта мы можем использовать встроенные датасеты
from surprise.dataset import BUILTIN_DATASETS
from surprise import SVD, KNNBasic, accuracy

Мы используем Dataset.load_from_file потому что surprise работает с форматом данных для рекомендаций
surprise — это библиотека для рекомендательных систем, и она ожидает данные в виде:

* user  item  rating  (timestamp)

То есть данные, где:

* user — ID пользователя

* item — ID товара (или фильма, книги и т.д.)

* rating — оценка (обычно число)

* timestamp — опционально (например, когда была поставлена оценка)

In [56]:
data = Dataset.load_from_file(
    'Data/u_df.txt', 
    reader=Reader(line_format='user item rating timestamp', # строки содержат user, item, rating, timestamp
    sep='\t')
)

Преобразуем данные к формату pandas DataFrame для удобной работы с ними:

In [57]:
u_df = pd.DataFrame(data.raw_ratings, columns=['userId', 'movieId', 'rating', 'timestamp'])
display(u_df.head(5))

,userId,movieId,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


В данных присутствуют следующие признаки:

* userId — идентификаторы пользователей сайта movielens;
* movieId — идентификаторы фильмов;
* rating — оценки фильмов, выставленные пользователями по шкале от 1 до 5;
* timestamp — время оценки фильма пользователем. Данный формат представления времени показывает, сколько секунд прошло с 1 января 1970 года.

Проверим, сколько уникальных фильмов в наборе данных

In [58]:
display(u_df['movieId'].nunique())

1682

Проверим, сколько уникальных пользователей в данных

In [59]:
display(u_df['userId'].nunique())

943

Проверим, какая оценка встречается в наборе данных чаще всего.

In [60]:
display(u_df['rating'].value_counts())

rating
4.0    34174
3.0    27145
5.0    21201
2.0    11370
1.0     6110
Name: count, dtype: int64

Библиотека surprise очень похожа на библиотеку sklearn, и тоже позволяет разбить данные на обучающую и тестовую выборки всего одной функцией — surprise.model_selection.train_test_split().

In [61]:
train, test = surprise.model_selection.train_test_split(data, test_size=0.25, random_state=13)

In [62]:
display(f'Тестовая выборка: {len(test)}')

'Тестовая выборка: 25000'

Импортируем функции для построения рекомендательных систем (SVD — для model-based-подхода и KNNBasic — для memory-basic-подхода) и для оценки качества результата.

Теперь реализуем обычную коллаборативную фильтрацию. Выберем оценку схожести через косинусную близость и item-based-подход:

In [63]:
sim_options = {
    'name': 'cosine',
    'user_based': False
}

# Модель для классификации по ближайшим соседям
knn = KNNBasic(sim_options=sim_options)
# Обучим алгоритм
knn.fit(train)

Computing the cosine similarity matrix...
Done computing similarity matrix.


Теперь посмотрим, какие рекомендации мы получили, с помощью следующей программы:

In [64]:
predictions = knn.test(test)
predictions

[Prediction(uid='7', iid='633', r_ui=5.0, est=4.199452349030111, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid='422', iid='287', r_ui=3.0, est=3.4703437660463736, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid='804', iid='163', r_ui=3.0, est=3.5716736533692854, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid='189', iid='480', r_ui=5.0, est=4.222825780855538, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid='238', iid='546', r_ui=3.0, est=3.473417286928204, details={'actual_k': 17, 'was_impossible': False}),
 Prediction(uid='804', iid='216', r_ui=4.0, est=3.922551907749182, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid='350', iid='204', r_ui=4.0, est=4.345238219480267, details={'actual_k': 38, 'was_impossible': False}),
 Prediction(uid='708', iid='993', r_ui=4.0, est=3.4458505791534115, details={'actual_k': 40, 'was_impossible': False}),
 Prediction(uid='193', iid='1078', r_ui=4.0, es

Информация о каждой паре будет содержать следующие характеристики:

* uid — id пользователя;
* iid — id элемента;
* r_ui (float) — реальный рейтинг, который этот пользователь поставил этому элементу;
* est (float) — предсказанный рейтинг.

Теперь посмотрим каков реальный рейтинг, выставленный с id 500 для фильма с id 699

In [65]:
real_rating = None
pred_rating = None

for pred in predictions:
    if pred.uid == '500' and pred.iid == '699':
        real_rating = pred.r_ui
        pred_rating = pred.est
        break
display(f'Пользователь: 500, поставил реальный рейтинг: {real_rating}, прогнозируемый рейтинг: {pred_rating:.2f}')

'Пользователь: 500, поставил реальный рейтинг: 3.0, прогнозируемый рейтинг: 3.47'

Теперь необходимо вычислить RMSE для получившихся предсказаний:

In [66]:
accuracy.rmse(predictions)

RMSE: 1.0272


1.0271678039029761

Если округлить результат до сотых, получаем 1.03.

Итак, мы построили систему рекомендаций и даже оценили её качество. Но как же вывести рекомендации для конкретного пользователя?

Для начала давайте оформим наши предсказания в таблицу и отсортируем их по прогнозируемому рейтингу:

In [67]:
pred = pd.DataFrame(predictions)
pred.sort_values(by=['est'], inplace=True, ascending=False)

Теперь мы можем вывести рекомендуемые для конкретного пользователя фильмы, начиная от наиболее релевантного (с точки зрения рекомендаций) и заканчивая наименее релевантным.

In [68]:
recommend = pred[pred.uid == '849']['iid'].to_list()
recommend

['234', '427', '568', '174']

# User based model

Теперь реализуем user-based-алгоритм. Определим какое значение RMSE получилось для коллаборативной фильтрации типа user-based. Ответ округлим до двух знаков после точки-разделителя.

In [69]:
sim_options = {
    'name': 'cosine',
    'user_based': True # Для user based подхода
}

# Модель для классификации по ближайшим соседям
knn = KNNBasic(sim_options=sim_options)
# Обучим алгоритм
knn.fit(train)

# Предсказания
predictions = knn.test(test)

# Метрика RMSE
rmse = accuracy.rmse(predictions)
display(f'RMSE user based: {rmse:.2f}')

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.0175


'RMSE user based: 1.02'

Теперь сравним полученные результаты с результатами SVD-алгоритма. Реализуем SVD с параметрами по умолчанию.

Ответ округлим до двух знаков после точки-разделителя.

In [70]:
# Алгоритм SVM
svd = SVD()
# обучение
svd.fit(train)

# Предсказания
predictions = svd.test(test)

# Метрика RMSE
rmse = accuracy.rmse(predictions)
display(f'RMSE user based: {rmse:.2f}')

RMSE: 0.9416


'RMSE user based: 0.94'

# Вывод: 
Cравнив все три подхода, можно с уверенностью сказать, что у SVD наилучшее качество, так как значение ошибки наименьшее - 0.94 RMSE.

# Гибридная модель рекомендательной системы

Импортируем нужные нам функции из этой библиотеки. На этом этапе сразу же загрузим инструменты оценки модели:

In [71]:
import pandas as pd
import numpy as np

# Импортируем класс LightFM — основная модель для рекомендательной системы
# Поддерживает различные функции потерь (logistic, BPR, WARP, WARP-kos)
from lightfm import LightFM

# Импортируем функцию для разбиения взаимодействий на обучающую и тестовую выборки
# random_train_test_split разделяет матрицу взаимодействий случайным образом
from lightfm.cross_validation import random_train_test_split

# Импортируем метрики качества для рекомендательной системы
# precision_at_k — вычисляет точность на первых k рекомендациях
# recall_at_k — вычисляет полноту на первых k рекомендациях
from lightfm.evaluation import precision_at_k, recall_at_k

# Для использования разряженных матриц
from scipy.sparse import csr_matrix

Работать мы будем с датасетом __goodreads_book__.

__Goodreads__ — это сайт, на котором люди могут добавлять книги в каталоги, искать их, изучать аннотации и отзывы. Пользователи также могут создавать сообщества, в которых они рекомендуют друг другу различную литературу, ведут блоги и устраивают обсуждения.

Подгрузим все файлы, относящиеся к этому набору данных:

In [72]:
ratings = pd.read_csv('Data/good_dread_books/ratings.csv') # Проставленные оценки
books = pd.read_csv('Data/good_dread_books/books.csv') # Инф. о книгах
tags = pd.read_csv('Data/good_dread_books/tags.csv') # Инф. о тегах
book_tags = pd.read_csv('Data/good_dread_books/book_tags.csv') # Книги с тегами

In [73]:
display(ratings.head(5))
print('--' * 40)
display(books.head(5))
print('--' * 40)
display(tags.head(5))
print('--' * 40)
display(book_tags.head(5))

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4


--------------------------------------------------------------------------------


,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


--------------------------------------------------------------------------------


,tag_id,tag_name
0,509,19th-century
1,923,20th-century
2,941,21st-century
3,1499,abuse
4,1540,action


--------------------------------------------------------------------------------


,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716


Сначала посмотрим на набор данных books: в этих данных есть обычный id книги, а есть id книги в системе Goodreads — этот id отображён в признаке goodreads_book_id. В других данных (book_tags) указан только id книги в системе Goodreads, поэтому нам необходимо добавить туда обычный id.

Добавим в набор данных book_tags признак с обычным id книги, используя соответствие обычного id и id в системе Goodreads и проверим какой обычный id у книги, которая имеет id 5 в системе Goodreads?

In [74]:
# Сделаем копии для удобства перед всеми манипуляциями
ratings_copy = ratings.copy()
books_copy = books.copy()
tags_copy = tags.copy()
book_tags_copy = book_tags.copy()

In [75]:
# Соединяем book_tags с books по goodreads_book_id, чтобы добавить book_id
book_tags_copy = book_tags_copy.merge(
    books_copy[['book_id', 'goodreads_book_id']],
    on='goodreads_book_id',
    how='left' # чтобы не потерять строки из book_tags
)

In [76]:
display(book_tags_copy[book_tags_copy['goodreads_book_id'] == 5])

,goodreads_book_id,tag_id,count,book_id
300,5,11557,40087,18
301,5,11305,39330,18
302,5,8717,17944,18
303,5,33114,12856,18
304,5,30574,11909,18
...,...,...,...,...
395,5,20781,299,18
396,5,32345,298,18
397,5,12600,282,18
398,5,3379,277,18


Далее необходимо будет оставить в наборе данных book_tags только те записи, теги для которых есть в данных tags.

Отфильтруем данные таким образом, чтобы в наборе данных book_tags остались только те строки, в которых находятся теги, информация о которых есть в наборе данных tags.

In [77]:
# Соединяем book_tags с tags по tags_id, чтобы оставить tags_id 
book_tags_copy = book_tags_copy.merge(
    tags_copy[['tag_id']], # нам от tags нужен только tag_id для фильтрации
    on='tag_id',
    how='inner' # inner оставит только совпавшие tag_id
)

In [78]:
display(book_tags_copy.shape[0])

300738

Таким образом мы подготовили информацию о тегах книг — это будет метаинформацией для построения рекомендательной системы. Теперь необходимо подготовить данные о взаимодействии пользователей и книг. Для этого понадобится файл ratings.

Оба набора данных (и про взаимодействия, и про метаинформацию) необходимо преобразовать в разрежённые матрицы. Это можно сделать с помощью специальной функции из модуля scipy:

```python
    !pip install scipy==1.10
    from scipy.sparse import csr_matrix
```

Примечание: Нам важно преобразовать данные в специальный формат, в котором хранятся разрежённые матрицы — будем использовать формат Compressed Sparse Row (CSR), подразумевающий подсчёт кумулятивной суммы количества элементов в строке вместо индексов строк.

Осуществляем преобразование следующим образом:

In [79]:
ratings_copy

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4
...,...,...,...
981751,10000,48386,5
981752,10000,49007,4
981753,10000,49383,5
981754,10000,50124,5


In [80]:
# Передаём в качестве аргументов в функцию выставленный рейтинг (это будут значения матрицы), 
# а также id пользователя и id книги (это будут индексы для строк и столбцов матрицы)
ratings_matrix = csr_matrix((ratings.rating, (ratings_copy.user_id, ratings_copy.book_id)))

Теперь нам необходимо составить матрицу с метаданными. В качестве индексов будут выступать id книги и id тега, и если у этой книги есть рассматриваемый тег, то на пересечении соответствующих строки и столбца будет выставлена единица.

In [81]:
book_tags_copy

,goodreads_book_id,tag_id,count,book_id
0,1,11305,37174,27
1,1,33114,12716,27
2,1,11743,9954,27
3,1,14017,7169,27
4,1,27199,3857,27
...,...,...,...,...
300733,33288638,9886,10,8892
300734,33288638,3358,10,8892
300735,33288638,1679,10,8892
300736,33288638,1659,9,8892


In [82]:
meta_matrix = csr_matrix(([1]*len(book_tags_copy), (book_tags_copy.book_id, book_tags_copy.tag_id)))

Теперь, чтобы проверить, что все сделано верно проверим каково среднее арифметическое значений разрежённой матрицы с рейтингами. Ответ округлии до трёх знаков после точки-разделителя.

In [83]:
display(f'Среднее арифметическое значений разряженной матрицы: {ratings_matrix.mean():.3f}')

'Среднее арифметическое значений разряженной матрицы: 0.007'

Отлично, данные подготовлены — теперь настало время определить модель, которую мы будем использовать. Сделаем это следующим образом:

In [84]:
model = LightFM(
    loss='warp-kos', # определяем функцию потерь
    random_state=42, # фиксируем значения
    learning_rate=0.05, # темп обучения
    no_components=100 # рамерность вектора для представления данных в модели
)

В качестве функции потерь мы выбрали значение 'warp', хотя, разумеется, это не единственный вариант. В модуле LightFM представлены следующие функции потерь:

* 'logistic' — логистическая функция. Полезна в случаях, когда есть как положительные, так и отрицательные взаимодействия, например 1 и -1.
* 'bpr' — байесовский персонализированный рейтинг. Можно применять, когда присутствуют только положительные взаимодействия.
* 'warp' — парный взвешенный приблизительный ранг. Используется, если необходимо повысить качество именно в верхней части списка рекомендаций.
* 'warp-kos' — модификация warp.

Разобьём данные на обучающую и тестовую выборки:

In [85]:
train, test = random_train_test_split(
    ratings_matrix, # Общая выборка
    test_percentage=0.2, # размер тестовой выборки 20%
    random_state=42
)

Теперь обучим модель на наших данных о взаимодействии, также используя метаданные о книгах. Для этого воспользуемся методом fit(). В этот метод передадим обучающую выборку, признаки товаров — item_features, количество эпох обучения (сколько раз мы будем показывать модели исходный датасет, чтобы она лучше выучила данные) — epochs, а также параметр verbose для отслеживания процесса обучения:

In [86]:
model = model.fit(
    train, # обучающая выборка (80 %)
    item_features=meta_matrix, # признаки товаров
    epochs=10, # количество эпох
    verbose=True, # отображение обучения
    num_threads=6 # количество используемых потоков
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9


Примечание: Обратите внимание: из-за трудоёмкости вычислений обучение модели и оценка качества могут занимать вплоть до 15-20 минут!

Теперь оценим качество полученной модели с помощью функции precision_at_k, передав в неё три аргумента: 
1. модель, 
2. тестовые данные 
3. обозначение метаданных (item_features = meta_matrix).

Примечание: Процесс расчёта метрик рекомендательной системы также является довольно затратным по времени. Для ускорения этого процесса вы можете передать параметр num_threads, чтобы указать количество потоков процессора, используемых для вычислений.

In [87]:
model_metrics = precision_at_k(
    model=model,
    test_interactions=test,
    item_features=meta_matrix,
    num_threads=6
)

In [88]:
# Выведим среднее арифметическое и округлим его до двух знаков после точки-разделителя.
metrics_array = model_metrics

# Считаем среднее по всем пользователям
mean_precision = np.mean(metrics_array)

# Округляем до двух знаков
mean_precision_rounded = round(mean_precision, 2)

# Выводим
print(mean_precision_rounded)

0.02


В рекомендательных системах метрики интерпретируются иначе, чем в задачах классификации. Показатели точности РС считаются хорошими, если они находятся в районе 0.1-0.3.

У нас получился не слишком высокий, но довольно неплохой результат. Чтобы его улучшить, можно попробовать следующее:

* Поработать над предобработкой данных, добавив в них дополнительную информацию о товарах. Также можно попробовать воспользоваться иным способом создания разреженной матрицы, например, форматом coo_matrix() или csc_matrix(), которые также входят в библиотеку scipy.
* Поиграться с параметрами модели LightFM — поуправлять темпом обучения (learning_rate), размерностью вектора для представления (no_components), количеством эпох обучения (epochs) и функцией потерь (loss).

Примечание. Для предсказания рейтинга нового пользователя можно воспользоваться методом predict():

```python
    scores = model.predict(<индекс интересующего пользователя>, np.arange(n_items), user_features=new_user_feature)
```